# Multi-Task QA — Spot Check

Compare **trained** vs **untrained** Qwen3-VL-2B-Instruct across all QA tasks:
position_qa, relposition_qa, near_gold_qa, gold_direction_qa, blue_line_qa, comparison_v1, direction_names.

- **Trained**: checkpoint-44000 (unmerged LoRA, merged at load time)
- **Untrained**: base Qwen3-VL-2B-Instruct from HuggingFace

In [ ]:
import os
import sys

os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import random
import torch
from transformers import AutoProcessor
from peft import PeftModel

try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
except ImportError:
    from transformers import AutoModelForVision2Seq

CHECKPOINT_DIR = "outputs/multi_task_dpo/checkpoint-44000"
BASE_MODEL = "Qwen/Qwen3-VL-2B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(42)
print("Working directory:", os.getcwd())

In [ ]:
from src.data.position_qa_generator import PositionQAGenerator
from src.data.relposition_qa_generator import RelpositionQAGenerator
from src.data.near_gold_qa_generator import NearGoldQAGenerator
from src.data.gold_direction_qa_generator import GoldDirectionQAGenerator
from src.data.blue_line_qa_generator import BlueLineQAGenerator
from src.data.comparison_v1_generator import ComparisonV1Generator
from src.data.direction_names_generator import DirectionNamesGenerator

TASKS = {
    "position_qa":      PositionQAGenerator(cross_axis_negative_prob=0.0),
    "relposition_qa":   RelpositionQAGenerator(),
    "near_gold_qa":     NearGoldQAGenerator(),
    "gold_direction_qa":GoldDirectionQAGenerator(),
    "blue_line_qa":     BlueLineQAGenerator(),
    "comparison_v1":    ComparisonV1Generator(),
    "direction_names":  DirectionNamesGenerator(),
}

N_SAMPLES = 8
task_samples = {}
for name, gen in TASKS.items():
    task_samples[name] = gen.generate_batch(N_SAMPLES)
    print(f"{name}: generated {len(task_samples[name])} samples")

In [ ]:
processor = AutoProcessor.from_pretrained(CHECKPOINT_DIR, trust_remote_code=True)
print("Processor loaded from", CHECKPOINT_DIR)

In [ ]:
base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.eval()
print("Base model loaded")

In [ ]:
trained_base = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
trained_model = PeftModel.from_pretrained(trained_base, CHECKPOINT_DIR)
trained_model = trained_model.merge_and_unload()
trained_model = trained_model.to(device)
trained_model.eval()
print("Trained model loaded and merged from", CHECKPOINT_DIR)

In [ ]:
def generate_response(model, processor, image, prompt, max_new_tokens=32):
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    ]
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")
    inputs = inputs.to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    generated = out[0][prompt_len:]
    return processor.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=True)

In [ ]:
def is_correct(response, chosen):
    r = response.strip().lower()
    c = chosen.strip().lower()

    # --- directional (left / right / up / down) ---
    if "left" in c and "right" not in c:
        return "left" in r and "right" not in r
    if "right" in c and "left" not in c:
        return "right" in r and "left" not in r
    if any(w in c for w in ["up", "above"]):
        return any(w in r for w in ["up", "above"])
    if any(w in c for w in ["down", "below"]):
        return any(w in r for w in ["down", "below"])

    # --- affirmative / negative ---
    _YES = ["yep", "absolutely", "certainly", "i think so", "uh-huh", "sure"]
    _NO  = ["nuh-uh", "nah", "certainly not", "absolutely not", "i don't think so"]
    if any(w in c for w in _YES):
        return any(w in r for w in _YES + ["yes"])
    if any(w in c for w in _NO) or c == "no":
        return any(w in r for w in _NO + ["no"])

    # --- CW / CCW / forward (relposition_qa, direction_names) ---
    if "counter-clockwise" in c or "ccw" in c or "counterclockwise" in c or "anticlock" in c:
        return "counter" in r or "ccw" in r or "anti" in r
    if "clockwise" in c or "cw" in c:
        return ("clockwise" in r or "cw" in r) and "counter" not in r and "anti" not in r
    if "forward" in c or "straight" in c or "full speed" in c:
        return "forward" in r or "straight" in r

    # --- action tokens ---
    if "<forward>" in c:
        return "<forward>" in response.strip()
    if "<anticlock>" in c:
        return "<anticlock>" in response.strip()
    if "<clock>" in c:
        return "<clock>" in response.strip() and "<anticlock>" not in response.strip()

    return c in r

## Run inference and compare (all tasks)

In [ ]:
from IPython.display import display

all_results = {}

for task_name, samples in task_samples.items():
    print("=" * 70)
    print(f"TASK: {task_name}")
    print("=" * 70)

    results = []
    for sample in samples:
        img = sample["image"]
        prompt = sample["prompt"]
        chosen = sample["chosen"]

        base_out = generate_response(base_model, processor, img, prompt)
        trained_out = generate_response(trained_model, processor, img, prompt)

        base_ok = is_correct(base_out, chosen)
        trained_ok = is_correct(trained_out, chosen)

        results.append({
            "prompt": prompt,
            "chosen": chosen,
            "base_out": base_out,
            "trained_out": trained_out,
            "base_correct": base_ok,
            "trained_correct": trained_ok,
        })

    base_acc = sum(r["base_correct"] for r in results) / len(results)
    trained_acc = sum(r["trained_correct"] for r in results) / len(results)
    print(f"Base model accuracy:    {base_acc:.1%} ({sum(r['base_correct'] for r in results)}/{len(results)})")
    print(f"Trained model accuracy: {trained_acc:.1%} ({sum(r['trained_correct'] for r in results)}/{len(results)})")
    print()

    for i, r in enumerate(results[:4]):
        print(f"--- Example {i+1} ---")
        print(f"Prompt:  {r['prompt']}")
        print(f"Correct: {r['chosen']}")
        display(samples[i]["image"])
        b_mark = "\u2713" if r["base_correct"] else "\u2717"
        t_mark = "\u2713" if r["trained_correct"] else "\u2717"
        print(f"  Base:    {r['base_out'][:120]} {b_mark}")
        print(f"  Trained: {r['trained_out'][:120]} {t_mark}")
        print()

    all_results[task_name] = results

## Summary

In [ ]:
print(f"{'Task':<20} {'Base':>8} {'Trained':>8}")
print("-" * 38)
for task_name, results in all_results.items():
    ba = sum(r["base_correct"] for r in results) / len(results)
    ta = sum(r["trained_correct"] for r in results) / len(results)
    print(f"{task_name:<20} {ba:>7.1%} {ta:>7.1%}")